# Simulate and reconstruct a small particle pack

Use the same simulation and reconstruction functions as `run.py` with a
small selection of reviewed particles. Run `prepare_data.ipynb` and label
extracted particles first, or point the configuration at an existing dataset.

Start with `uv run --extra notebooks jupyter lab`. A CUDA-capable GPU is
required to build particle SDFs in the current workflow, including when the
simulation device is set to `cpu`. The default device is `cuda:0`. Results are written to a new timestamped
directory. The optional Rerun viewer records the simulated motion.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src").is_dir()
)
os.chdir(PROJECT_ROOT)


In [ ]:
from dataclasses import asdict
from datetime import datetime

import newton.viewer
import numpy as np
from run import load_correct_ids
from src.simulator import NewtonSimulator
from src.workflow import SimulationConfig, simulate_particle_pack
from src.simulation_pipeline import add_noise, to_volume_parallel


## Select reviewed particles

Adjust the input paths if you use a different dataset. Keep `N_OBJECTS`
small for the first run. Container dimensions, object scale, and duration
are starting points; larger particles or more objects may need more space
and simulation time.

In [ ]:
PARTICLE_DIR = Path("data/toy_data_GOH_6_250/sim_data_uint8")
LABELS_PATH = Path("data/toy_data_GOH_6_250/manual_check.json")
DEVICE = "cuda:0"
N_OBJECTS = 10
SEED = 42
SHOW_VIEWER = False
OUTPUT_SHAPE = (512, 512, 512)
OUTPUT_DIR = Path("data/toy_data_GOH_6_250/outputs") / datetime.now().strftime("demo_%Y%m%d_%H%M%S_%f")

config = SimulationConfig(
    n_frames=1200,
    obj_scale=0.4,
    container_radius=2.0,
    container_height=5.0,
    seed=SEED,
)


In [ ]:
correct_ids = load_correct_ids(LABELS_PATH)
assert len(correct_ids) >= N_OBJECTS, "Label more correct particles or reduce N_OBJECTS."
rng = np.random.default_rng(SEED)
obj_ids = [int(i) for i in rng.choice(correct_ids, size=N_OBJECTS, replace=False)]
for obj_id in obj_ids:
    for extension in ("obj", "npy.gz"):
        assert (PARTICLE_DIR / f"{obj_id}.{extension}").is_file()
print(f"Selected IDs: {obj_ids}")


## Run Newton simulation

In [ ]:
viewer = newton.viewer.ViewerRerun(keep_historical_data=True) if SHOW_VIEWER else None
results = simulate_particle_pack(
    obj_ids=obj_ids,
    obj_dir=PARTICLE_DIR,
    sim_config=config,
    device=DEVICE,
    viewer=viewer,
)
if viewer is not None:
    viewer.show_notebook()


## Save transforms and check particle locations

The built-in check tests particle centers against the container radius and
floor. It does not establish that the pack has settled or that every voxel
fits inside the reconstruction volume. Inspect the resulting slices too.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NewtonSimulator.save_simulation_results(
    results, OUTPUT_DIR / "sim_res.json", metadata=asdict(config)
)
valid = NewtonSimulator.verify_simulation_correctness(
    results, container_radius=config.container_radius
)
print(f"Particle centers pass bounds check: {valid}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


## Reconstruct volumes and add noise

Reconstruction writes `tomo.zarr` and `mask.zarr`; the latter stores particle
IDs as integer labels. A single worker keeps this example usable in Jupyter.
Adjust `OUTPUT_SHAPE` to contain your simulated pack; reconstruction clips
voxels outside that shape. Rerunning replaces the volume stores in this run.

In [ ]:
assert valid, "Inspect the simulation and adjust its parameters before reconstruction."
to_volume_parallel(
    simulation_results=results,
    obj_ids=obj_ids,
    obj_dir=str(PARTICLE_DIR),
    output_dir=str(OUTPUT_DIR),
    output_shape=OUTPUT_SHAPE,
    obj_scale=config.obj_scale,
    parallel=1,
    metadata={"simulation_config": asdict(config), "seed": SEED},
)
add_noise(
    input_path=str(OUTPUT_DIR / "tomo.zarr"),
    output_path=str(OUTPUT_DIR / "tomo_noisy.zarr"),
    noise_std=15,
    seed=SEED,
)
print(f"Open inspect_results.ipynb and set RUN_DIR to {str(OUTPUT_DIR)!r}.")
